# Türkiye'de Depremlerin Coğrafi Dağılımı, Derinliği ve Büyüklüğü (1994–2023)

**Çalışma sorusu:** Türkiye'de geçmiş yıllarda meydana gelen depremlerin coğrafi dağılımı, derinliği ve büyüklükleri zaman içerisinde bölgesel olarak nasıl bir değişim göstermektedir?

**Veri:** Türkiye Deprem Veri Seti (`veriler.csv`) — Kaggle, *ozgecinko/turkey-earthquake-data-1914-2023*. Lisans: CC BY-SA 4.0.

**Akış:** veri yükleme → temizleme → keşifsel analiz → bulgular → sitede kullanılacak özet dosyaların dışa aktarımı.

> **Yapay zekâ kullanım beyanı:** Bu notebook'un kod iskeleti Claude (Anthropic) ile birlikte hazırlanmıştır. Bulguların yorumları ve hikâye metni bana aittir. *(Kendi cümlelerinle güncelle.)*

## 1. Kurulum ve veri yükleme

In [ ]:
import os, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110

CIKTI = "site_data"            # sitede kullanılacak özet dosyalar buraya yazılır
os.makedirs(CIKTI, exist_ok=True)

M_ESIK = 4.0                   # zaman karşılaştırmalarında kullanılan büyüklük eşiği
SABIT_DERINLIKLER = [0.0, 5.0, 10.0]   # varsayılan/sabit atanmış olma şüphesi olan derinlikler (km)
KISMI_YILLAR = [1994, 2023]    # veri 11 Eylül 1994'te başlıyor, 31 Temmuz 2023'te bitiyor

In [ ]:
# Dosya notebook ile aynı klasörde değilse (Colab) yükleme penceresi açılır.
aday = ["veriler.csv", "veriler_1.csv"]
yol = next((a for a in aday if os.path.exists(a)), None)
if yol is None:
    from google.colab import files
    yuklenen = files.upload()
    yol = list(yuklenen.keys())[0]

ham = pd.read_csv(yol)
print(yol, ham.shape)
ham.head()

## 2. İlk bakış

In [ ]:
print(ham.dtypes)
print()
print("Eksik değer oranları:")
print(ham.isna().mean().round(3))
print()
print("Tip sütunu:")
print(ham["Tip"].value_counts())
print()
print(ham[["Enlem", "Boylam", "Der(km)", "xM", "MD", "ML", "Mw", "Ms", "Mb"]].describe().T)

**Gözlemler (kendi cümlelerinle tamamla):**
- Veri 50.000 satır; xM en az 3,0 → kayıt yalnızca büyüklüğü 3,0 ve üzeri depremleri içeriyor.
- MD, ML, Ms, Mb sütunlarında 0,0 değerleri "ölçülmedi" anlamına geliyor gibi görünüyor; Mw'nin ise büyük kısmı boş.
- …

## 3. Temizleme

Her karar gerekçesiyle birlikte yazılmıştır (Yöntem bölümünde de kullanılacak).

In [ ]:
df = ham.rename(columns={
    "Olus tarihi": "tarih_str", "Olus zamani": "zaman_str",
    "Enlem": "enlem", "Boylam": "boylam", "Der(km)": "derinlik", "xM": "mag",
}).copy()
n_baslangic = len(df)

# 3.1 Tarih: "2023.07.31" biçimini gerçek tarihe çevir
df["tarih"] = pd.to_datetime(df["tarih_str"], format="%Y.%m.%d", errors="coerce")
print("Ayrıştırılamayan tarih sayısı:", df["tarih"].isna().sum())
df = df.dropna(subset=["tarih"]).copy()
df["yil"] = df["tarih"].dt.year
df["ay"] = df["tarih"].dt.month

# 3.2 Yinelenen kayıtlar (Deprem Kodu tekil olmalı)
yinelenen = df["Deprem Kodu"].duplicated().sum()
print("Yinelenen Deprem Kodu:", yinelenen)
df = df.drop_duplicates(subset="Deprem Kodu").copy()

# 3.3 Ölçülmeyen büyüklük türleri 0,0 ile kodlanmış → eksik değer (NaN) yap
for c in ["MD", "ML", "Mw", "Ms", "Mb"]:
    df[c] = df[c].replace(0.0, np.nan)
# Ana büyüklük ölçüsü olarak xM kullanılıyor: eksiksiz ve her kayıtta dolu.
print("xM eksik sayısı:", df["mag"].isna().sum())

# 3.4 Koordinatlar makul aralıkta mı?
print("Enlem aralığı:", df["enlem"].min(), df["enlem"].max())
print("Boylam aralığı:", df["boylam"].min(), df["boylam"].max())

In [ ]:
# 3.5 "Tip" sütunu: 114 satır "Sm". Anlamı veri belgelerinde net değil, hepsi 3,0–3,5 arası.
# Yalnızca "Ke" (deprem) kayıtları tutuluyor; etkisi çok küçük (%0,2).
sm = (df["Tip"] != "Ke").sum()
print("Çıkarılan Tip != 'Ke' satır sayısı:", sm)
df = df[df["Tip"] == "Ke"].copy()

In [ ]:
# 3.6 Derinlik: tek değerlerde şüpheli yığılma var (özellikle 5,0 km).
# Bu değerler ölçülmüş değil, varsayılan olarak atanmış olabilir (hipotez).
vc = df["derinlik"].round(1).value_counts().head(10)
print(vc)

fig, ax = plt.subplots(figsize=(7, 3))
vc.sort_index().plot(kind="bar", ax=ax, color="#888")
ax.set_title("En sık görülen derinlik değerleri")
ax.set_xlabel("Derinlik (km)"); ax.set_ylabel("Kayıt sayısı")
plt.tight_layout(); plt.show()

df["d_ok"] = ~df["derinlik"].isin(SABIT_DERINLIKLER)
print("Derinliği 'güvenilir' sayılan oran:", round(df["d_ok"].mean(), 3))

**Karar:** Derinlik analizlerinde 0, 5 ve 10 km kayıtları dışarıda bırakılıyor (`d_ok = False`). Bu kayıtlar haritada ve büyüklük/sayı analizlerinde kullanılmaya devam ediyor. Bunu Yöntem bölümünde ve sınırlılıklarda açıkça yaz.

In [ ]:
# 3.7 İl ve bölge: konum yalnızca "Yer" metninin içinde, ör. "DOGANLI-PAZARCIK (KAHRAMANMARAS) [East 1.3 km]"
TR_MAP = str.maketrans("ÇĞÖŞÜ", "CGOSU")

def normalize(s):
    s = s.replace("İ", "I").replace("ı", "I").upper()
    return s.translate(TR_MAP).strip()

def yer_adi(yer):
    ic = re.search(r"\(([^)]+)\)", yer)
    ad = ic.group(1) if ic else re.sub(r"\[.*?\]", "", yer)
    return normalize(ad)

BOLGELER = {
    "Marmara": ["BALIKESIR", "BILECIK", "BURSA", "CANAKKALE", "EDIRNE", "ISTANBUL",
                "KIRKLARELI", "KOCAELI", "SAKARYA", "TEKIRDAG", "YALOVA"],
    "Ege": ["AFYONKARAHISAR", "AYDIN", "DENIZLI", "IZMIR", "KUTAHYA", "MANISA", "MUGLA", "USAK"],
    "Akdeniz": ["ADANA", "ANTALYA", "BURDUR", "HATAY", "ISPARTA", "KAHRAMANMARAS", "MERSIN", "OSMANIYE"],
    "İç Anadolu": ["ANKARA", "AKSARAY", "CANKIRI", "ESKISEHIR", "KARAMAN", "KAYSERI", "KIRIKKALE",
                   "KIRSEHIR", "KONYA", "NEVSEHIR", "NIGDE", "SIVAS", "YOZGAT"],
    "Karadeniz": ["AMASYA", "ARTVIN", "BARTIN", "BAYBURT", "BOLU", "CORUM", "DUZCE", "GIRESUN",
                  "GUMUSHANE", "KARABUK", "KASTAMONU", "ORDU", "RIZE", "SAMSUN", "SINOP", "TOKAT",
                  "TRABZON", "ZONGULDAK"],
    "Doğu Anadolu": ["AGRI", "ARDAHAN", "BINGOL", "BITLIS", "ELAZIG", "ERZINCAN", "ERZURUM",
                     "HAKKARI", "IGDIR", "KARS", "MALATYA", "MUS", "TUNCELI", "VAN"],
    "Güneydoğu Anadolu": ["ADIYAMAN", "BATMAN", "DIYARBAKIR", "GAZIANTEP", "KILIS", "MARDIN",
                          "SIIRT", "SANLIURFA", "SIRNAK"],
}
IL_BOLGE = {il: b for b, iller in BOLGELER.items() for il in iller}
print("Tanımlı il sayısı:", len(IL_BOLGE), "(81 olmalı)")

DENIZLER = {"AKDENIZ", "DOGU AKDENIZ", "EGE DENIZI", "MARMARA DENIZI", "KARADENIZ"}
GOLLER = {"VAN GOLU": "VAN", "KUS GOLU": "BALIKESIR"}   # göl adıyla yazılmış kayıtlar ilgili ile bağlanır

df["il"] = df["Yer"].apply(yer_adi).replace(GOLLER)

def bolge_bul(ad):
    ad = GOLLER.get(ad, ad)
    if ad in IL_BOLGE: return IL_BOLGE[ad]
    if ad in DENIZLER: return "Deniz"
    return "Yurt dışı / sınır"

df["bolge"] = df["il"].apply(bolge_bul)
print(df["bolge"].value_counts())
print()
print("'Yurt dışı / sınır' olarak etiketlenen adlar (kontrol et):")
print(df.loc[df["bolge"] == "Yurt dışı / sınır", "il"].value_counts().head(25))

**Kontrol:** Yukarıdaki "Yurt dışı / sınır" listesinde yanlış eşleşmiş bir il adı (yazım farkı) görürsen `IL_BOLGE` sözlüğüne veya `normalize` fonksiyonuna ekle, hücreyi yeniden çalıştır.

In [ ]:
# 3.8 Dönemler (üç eşit 10 yıllık dilim; ilk ve son dönem kısmi)
df["donem"] = pd.cut(df["yil"], bins=[1993, 2003, 2013, 2023],
                     labels=["1994–2003", "2004–2013", "2014–2023"])
df["kismi_yil"] = df["yil"].isin(KISMI_YILLAR)

print(f"Başlangıç: {n_baslangic:,} satır → temiz: {len(df):,} satır")
print("Tarih aralığı:", df["tarih"].min().date(), "→", df["tarih"].max().date())

## 4. Keşifsel analiz

### 4.1 Büyüklük eşiği neden gerekli?
Küçük depremler eski yıllarda daha az tespit edilmiş olabilir. Yıllık sayıları farklı eşiklerle karşılaştırıyoruz.

In [ ]:
yillik = pd.DataFrame({
    "n_m3": df.groupby("yil").size(),
    "n_m35": df[df.mag >= 3.5].groupby("yil").size(),
    "n_m4": df[df.mag >= 4.0].groupby("yil").size(),
    "n_m5": df[df.mag >= 5.0].groupby("yil").size(),
}).fillna(0).astype(int)
yillik["max_mag"] = df.groupby("yil")["mag"].max()
yillik["kismi_yil"] = yillik.index.isin(KISMI_YILLAR)

fig, ax = plt.subplots(figsize=(9, 4))
for c, lab in [("n_m3", "M ≥ 3,0"), ("n_m35", "M ≥ 3,5"), ("n_m4", "M ≥ 4,0")]:
    ax.plot(yillik.index, yillik[c], label=lab)
ax.set_yscale("log"); ax.legend()
ax.set_title("Yıllık deprem sayısı, farklı büyüklük eşikleri (log ölçek)")
ax.set_xlabel("Yıl"); ax.set_ylabel("Deprem sayısı")
plt.tight_layout(); plt.show()
yillik

**Yorum (kendi cümlelerinle):** Düşük eşikte sayılar yıllara göre çok dalgalı. 2023 artışı büyük ölçüde 6 Şubat artçılarından kaynaklanıyor olmalı. Bu yüzden zaman karşılaştırmasında M ≥ 4,0 kullanıyoruz.

### 4.2 Coğrafi dağılım

In [ ]:
m4 = df[df.mag >= M_ESIK].copy()
print("M ≥ 4,0 deprem sayısı:", len(m4))

fig, ax = plt.subplots(figsize=(10, 5))
for d, renk in zip(["1994–2003", "2004–2013", "2014–2023"], ["#bbbbbb", "#7aa6c2", "#d1495b"]):
    s = m4[m4["donem"] == d]
    ax.scatter(s.boylam, s.enlem, s=(s.mag - 3) ** 3 * 2, alpha=.5, c=renk, label=d, linewidths=0)
ax.set_xlabel("Boylam"); ax.set_ylabel("Enlem"); ax.legend()
ax.set_title("M ≥ 4,0 depremlerin dağılımı (nokta boyutu = büyüklük)")
ax.set_aspect(1.25)
plt.tight_layout(); plt.show()

### 4.3 Bölge ve dönem karşılaştırması (M ≥ 4,0)

In [ ]:
sira = ["Marmara", "Ege", "Akdeniz", "İç Anadolu", "Karadeniz", "Doğu Anadolu",
        "Güneydoğu Anadolu", "Deniz", "Yurt dışı / sınır"]
tablo = (m4.pivot_table(index="bolge", columns="donem", values="mag", aggfunc="size", fill_value=0, observed=False)
           .reindex(sira))
tablo["Toplam"] = tablo.sum(axis=1)
tablo["Pay (%)"] = (tablo["Toplam"] / tablo["Toplam"].sum() * 100).round(1)
tablo

### 4.4 Derinlik

In [ ]:
dd = df[df.d_ok].copy()
print("Derinlik analizinde kullanılan kayıt:", len(dd), "/", len(df))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(dd["derinlik"].clip(upper=60), bins=60, color="#7aa6c2")
ax.set_xlabel("Derinlik (km, 60'ta kırpılmış)"); ax.set_ylabel("Kayıt sayısı")
ax.set_title("Derinlik dağılımı (sabit değerler hariç)")
plt.tight_layout(); plt.show()

def ozet(g):
    return pd.Series({
        "n": len(g),
        "medyan_derinlik": g["derinlik"].median(),
        "p10": g["derinlik"].quantile(.10), "q1": g["derinlik"].quantile(.25),
        "q3": g["derinlik"].quantile(.75), "p90": g["derinlik"].quantile(.90),
        "pay_30km_ustu": (g["derinlik"] > 30).mean(),
    })

derinlik_bolge = dd.groupby("bolge").apply(ozet, include_groups=False).reindex(sira).dropna(how="all")
derinlik_bolge.round(2)

## 5. Bulgular

Aşağıdaki hücreler hikâyede kullanabileceğin sayıları üretir. **Hangi 3–5 bulguyu seçeceğin ve nasıl yorumlayacağın sana ait.** Korelasyon ile nedensellik farkına dikkat et.

In [ ]:
bulgu = {}
bulgu["toplam_kayit_temiz"] = int(len(df))
bulgu["m4_toplam"] = int(len(m4))
bulgu["tarih_baslangic"] = str(df["tarih"].min().date())
bulgu["tarih_bitis"] = str(df["tarih"].max().date())

# En çok M>=4 deprem yaşayan iller ve bölgeler
il_say = m4[~m4.bolge.isin(["Deniz", "Yurt dışı / sınır"])]["il"].value_counts()
print("En çok M≥4 deprem olan 10 il:"); print(il_say.head(10)); print()
bulgu["en_cok_m4_iller"] = {k: int(v) for k, v in il_say.head(10).items()}

# 2023'ün payı
pay_2023 = (m4["yil"] == 2023).mean()
print(f"M≥4 depremlerin %{pay_2023*100:.1f}'i 2023 yılında (Ocak–Temmuz).")
bulgu["m4_pay_2023"] = round(float(pay_2023), 3)

# Dönemler arası bölgesel payın değişimi
pay = m4[m4.bolge.isin(sira[:7])].pivot_table(index="bolge", columns="donem", values="mag",
                                               aggfunc="size", fill_value=0, observed=False)
pay = (pay / pay.sum() * 100).round(1).reindex(sira[:7])
print("\nBölgelerin dönem içindeki payı (%, M≥4):"); print(pay)
bulgu["bolge_pay_donem"] = pay.to_dict()

# En büyük 10 deprem
en_buyuk = df.nlargest(10, "mag")[["tarih", "enlem", "boylam", "derinlik", "mag", "il", "bolge"]]
print("\nEn büyük 10 deprem:"); print(en_buyuk)

**Seçtiğim bulgular ve yorumlarım (kendin yaz):**
1. …
2. …
3. …

## 6. Sitede kullanılacak özet dosyaların dışa aktarımı

Site ham veriyi değil bu küçük dosyaları okuyacak (toplam < 5 MB şartı).

In [ ]:
# 6.1 Haritadaki noktalar (M ≥ 4,0), tarihe göre sıralı
m4s = m4.sort_values("tarih")
noktalar = [
    {"t": r.tarih.strftime("%Y-%m-%d"), "lat": round(r.enlem, 3), "lon": round(r.boylam, 3),
     "d": round(r.derinlik, 1), "d_ok": bool(r.d_ok), "m": round(r.mag, 1), "b": r.bolge, "il": r.il}
    for r in m4s.itertuples()
]
with open(f"{CIKTI}/deprem_noktalar_m4.json", "w", encoding="utf-8") as f:
    json.dump(noktalar, f, ensure_ascii=False, separators=(",", ":"))

# 6.2 Yıllık sayılar
yillik.reset_index().to_csv(f"{CIKTI}/yillik_sayilar.csv", index=False)

# 6.3 Bölge × yıl (sıfırlar dahil, çizgi grafik için)
bolgeler7 = sira[:7]
idx = pd.MultiIndex.from_product([sira, range(1994, 2024)], names=["bolge", "yil"])
by = pd.DataFrame(index=idx)
by["n_m3"] = df.groupby(["bolge", "yil"]).size()
by["n_m4"] = m4.groupby(["bolge", "yil"]).size()
by["max_mag"] = df.groupby(["bolge", "yil"])["mag"].max()
by["medyan_derinlik"] = dd.groupby(["bolge", "yil"])["derinlik"].median()
by[["n_m3", "n_m4"]] = by[["n_m3", "n_m4"]].fillna(0).astype(int)
by = by.reset_index()
by["kismi_yil"] = by["yil"].isin(KISMI_YILLAR)
by.round(2).to_csv(f"{CIKTI}/bolge_yil.csv", index=False)

# 6.4 İl × yıl (okuyucu keşfi için; yalnızca var olan kombinasyonlar, eksikler 0 sayılmalı)
iy = df.groupby(["il", "bolge", "yil"]).agg(n_m3=("mag", "size"), max_mag=("mag", "max")).reset_index()
iy4 = m4.groupby(["il", "yil"]).size().rename("n_m4").reset_index()
iy = iy.merge(iy4, on=["il", "yil"], how="left").fillna({"n_m4": 0})
iy["n_m4"] = iy["n_m4"].astype(int)
iy.to_csv(f"{CIKTI}/il_yil.csv", index=False)

# 6.5 Derinlik özetleri
derinlik_bolge.round(2).reset_index().to_csv(f"{CIKTI}/derinlik_bolge.csv", index=False)
dbd = dd.groupby(["bolge", "donem"], observed=True).apply(ozet, include_groups=False).reset_index()
dbd.round(2).to_csv(f"{CIKTI}/derinlik_bolge_donem.csv", index=False)

# 6.6 Bulgular özeti
with open(f"{CIKTI}/bulgular.json", "w", encoding="utf-8") as f:
    json.dump(bulgu, f, ensure_ascii=False, indent=1)

# 6.7 Boyut kontrolü (toplam 5 MB'ı geçmemeli)
toplam = 0
for ad in sorted(os.listdir(CIKTI)):
    b = os.path.getsize(os.path.join(CIKTI, ad)); toplam += b
    print(f"{ad:32s} {b/1024:8.1f} KB")
print(f"\nToplam: {toplam/1024/1024:.2f} MB  (sınır: 5 MB)")

## 7. Dosyaları indirme (Colab)

In [ ]:
import shutil
shutil.make_archive("site_data", "zip", CIKTI)
try:
    from google.colab import files
    files.download("site_data.zip")
except ImportError:
    print("Colab dışında çalışıyor; site_data.zip bu klasörde.")

## 8. Sınırlılıklar (Yöntem bölümü için not al)

- Veri 11.09.1994 – 31.07.2023 arasını kapsıyor; 1994 ve 2023 kısmi yıl.
- Yalnızca büyüklüğü ≥ 3,0 olan kayıtlar var; zaman karşılaştırmalarında bu yüzden M ≥ 4,0 eşiği kullanıldı.
- Derinlik değerlerinde 0, 5 ve 10 km'de yığılma var; derinlik analizinde bu kayıtlar dışarıda bırakıldı.
- Bölge bilgisi `Yer` metninden çıkarıldı; il adı yazım farkları normalize edildi. Deniz ve yurt dışı kayıtlar ayrı kategori.
- Gözlenen ilişkiler nedensellik göstermez.